# Tariff example usage
This notebook demonstrates how to use the `Tariff` class to calculate energy costs based on a given tariff structure. We will create a sample tariff and then use it to compute the cost of energy consumption over a specified period.

### 1. Creating an index
Most tariffs are based on one or more indexes. For example, a common structure is to have a fixed cost component and a variable cost component that depends on the day-ahead electricity price. Let's create an index for the day-ahead price using the `EntsoeDayAheadIndex`:

In [2]:
from os import environ

from energy_cost.index import EntsoeDayAheadIndex, Index

Index.register("Belpex15min", EntsoeDayAheadIndex(country_code="BE", api_key=environ["ENTSOE_API_KEY"]))

### 2. Defining a tariff
The easiest way to define a tariff is to specify it in a YAML file. This allows for a clear separation of the tariff definition from the code that uses it. you can find some real world examples in `data/tariffs/`. 

The yaml structure is as follows:

```yaml
supplier: "Foo" # The energy supplier providing the tariff
product: "Bar" # The specific product or plan offered by the supplier
by_meter_type: 
  single_rate: # The metering type, eg. single_rate, tou_peak, tou_offpeak, ...
  # The ordered list of price formulas that apply to this metering type. The formulas are assumed to be valid from their start date until the start date of the next formula, or indefinitely if there is no next formula.
  - start: 2025-01-01T00:00:00+01 # The start date of the formula, in ISO 8601 format
    formula:
      constant_cost: 1.625 # The fixed cost component
      variable_costs: # The variable cost component depends on one or more indexes, which are multiplied by a scalar and summed to calculate the cost
      - index: Belpex15min # The index used for the variable cost
        scalar: 0.108 # The scalar applied to the index to calculate the variable cost
      - index: BelpexRLPO # Another index used in the formula
        scalar: 0.05 
```

In [3]:
from energy_cost.tariff import Tariff

tariff = Tariff.from_yaml("../data/tariffs/EBEM/Groen_Dynamic.yml")
tariff

Tariff(supplier='EBEM', product='Groen Dynamic', by_meter_type={<MeterType.SINGLE_RATE: 'single_rate'>: [TimedPriceFormula(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), formula=PriceFormula(constant_cost=1.625, variable_costs=[IndexAdder(index='Belpex15min', scalar=0.108)]))], <MeterType.INJECTION: 'injection'>: [TimedPriceFormula(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), formula=PriceFormula(constant_cost=-1.1, variable_costs=[IndexAdder(index='Belpex15min', scalar=0.0925)]))]})

### 3. Calculating cost per energy consumption in a given period
Once we have defined our tariff, we can use it to calculate the cost of energy consumption over a specified period. This is done by calling the `get_cost` method of the `Tariff` class, which returns a DataFrame with the cost for each time step in the given period.

In [10]:
import datetime as dt

from energy_cost.tariff import MeterType

start = dt.datetime.fromisoformat("2026-03-08 00:00:00+01:00")
end = dt.datetime.fromisoformat("2026-03-10 00:00:00+01:00")
resolution = dt.timedelta(minutes=5)
tariff.get_cost(MeterType.INJECTION, start, end, resolution)

,timestamp,value
0,2026-03-08 00:00:00+01:00,11.546600
1,2026-03-08 00:05:00+01:00,11.546600
2,2026-03-08 00:10:00+01:00,11.546600
3,2026-03-08 00:15:00+01:00,10.796425
4,2026-03-08 00:20:00+01:00,10.796425
...,...,...
571,2026-03-09 23:35:00+01:00,10.880600
572,2026-03-09 23:40:00+01:00,10.880600
573,2026-03-09 23:45:00+01:00,11.048025
574,2026-03-09 23:50:00+01:00,11.048025
